                # BootstrapFewShotWithRandomSearch (BootstrapRS)

                Build several bootstrapped demo sets and choose among them using the frozen validation split.

                **Use it when:** BootstrapFewShot is promising and you can afford multiple candidate programs to reduce dependence on one demo sample.

                **What compilation changes:** Demonstrations only, but candidate selection adds a validation-driven search loop.

                Important configuration in this benchmark:

                - 8 candidate programs in the full profile (2 in smoke)
- two bootstrapped plus two labeled demos per candidate
- single evaluation thread by default

                The 74-row dataset and pair-grouped train/validation/test membership are frozen.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import (
    artifact_paths,
    benchmark_snapshot,
    learned_program_preview,
    verify_prompt_artifact,
)

OPTIMIZER = 'bootstrap-random-search'
RUN_MODE = os.getenv("CHAPTER06_RUN", "inspect").lower()
print(f"mode={RUN_MODE!r}; optimizer={OPTIMIZER!r}")
print("safe default: inspect checked-in full-run artifacts; no API calls")

mode='inspect'; optimizer='bootstrap-random-search'
safe default: inspect checked-in full-run artifacts; no API calls


                ## Compile shape

                This is the essential DSPy call used by the shared runner (setup variables omitted):

                ```python
                optimizer = dspy.BootstrapFewShotWithRandomSearch(
    metric=exact_match, num_candidate_programs=profile.bootstrap_candidates,
    max_bootstrapped_demos=2, max_labeled_demos=2, num_threads=1,
)
optimized_detector = optimizer.compile(detector, trainset=trainset, valset=valset)
                ```

                `compile` returns a program. Calling that program on the untouched test examples is
                a separate phase; the notebook reports optimization cost/time separately from inference latency.

In [2]:
print(benchmark_snapshot(OPTIMIZER))
print()
print(artifact_paths(OPTIMIZER))

BootstrapRS — frozen full-profile rerun
status: completed
task model: openai/gpt-5.6-luna
test accuracy: 70.0%
uplift: +20.0 points vs Luna reference
optimization: $0.2860 and 6.6min
inference latency: mean 1.79s; p95 2.66s
reload checks: prompt=True; model=None; predictions=3/3 labels
complete run: chapter06/results/runs/full/bootstrap-random-search/20260715T021734.417538Z

Complete artifacts:
- complete_output: chapter06/results/runs/full/bootstrap-random-search/20260715T021734.417538Z/complete_output.txt
- lm_history: chapter06/results/runs/full/bootstrap-random-search/20260715T021734.417538Z/lm_history.jsonl
- metrics: chapter06/results/runs/full/bootstrap-random-search/20260715T021734.417538Z/metrics.json
- predictions: chapter06/results/runs/full/bootstrap-random-search/20260715T021734.417538Z/predictions.jsonl
- program: chapter06/results/runs/full/bootstrap-random-search/20260715T021734.417538Z/program.json
- prompts: chapter06/results/runs/full/bootstrap-random-search/20260715

## Read the result

The extra compile spend buys candidate selection, not a new instruction. Check whether the held-out gain justifies the search relative to plain BootstrapFewShot.

The next cell shows a bounded readable preview. The complete, lossless prompt and
serialized program remain at the paths printed above.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 2
1. is_ai=False: We've covered the "CR" of CRUD. Now let's move on to the "U" (Update). Updating a resource is very similar to creating a resource. They are both multi-step processes. First, the…
2. is_ai=True: Having covered the “CR” in CRUD, we now turn to “U” for Update. Updating a resource closely resembles creating one, for both are multi-step affairs. First, the user requests a f…

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt_state_equal': True, 'mismatches': []}


## Run it yourself (explicit opt-in)

The default `inspect` mode is offline and free. For a live run, first install from
the repository root with `python -m pip install -r requirements.txt`, configure
`OPENAI_API_KEY`, and restart the kernel.

- Bounded code-path check: launch Jupyter with `CHAPTER06_RUN=smoke`.
- Complete frozen-split rerun: launch Jupyter with `CHAPTER06_RUN=full`.

A full run is intentionally not triggered by opening or choosing “Run All”: it can
take minutes or incur model charges. The weight optimizers download and train a
small Qwen model locally through MPS/CPU and require the optional training stack. Live
artifacts are written to `chapter06/results/runs/<profile>/<optimizer>/<run-id>/`.
Rebuild the comparison afterward with `python -m chapter06.summarize_optimizer_results`.

In [4]:
if RUN_MODE == "inspect":
    print("Live run skipped. Set CHAPTER06_RUN=smoke or CHAPTER06_RUN=full explicitly.")
elif RUN_MODE in {"smoke", "full"}:
    from chapter06.optimizer_experiment import run_experiment

    live_result = run_experiment(OPTIMIZER, profile_name=RUN_MODE)
    print(live_result)
else:
    raise ValueError("CHAPTER06_RUN must be inspect, smoke, or full")

Live run skipped. Set CHAPTER06_RUN=smoke or CHAPTER06_RUN=full explicitly.
